# Nested Grid Preprocessing: Bathymetry Extraction & Visualization

---

## Overview

When using **SWAN nested grids**, both the coarse (parent) and fine (child) grids need bathymetry. In our case, both grids read from the same GEBCO bathymetry file, and SWAN handles the interpolation internally. However, for quality control and visualization, this notebook:

1. **Loads the GEBCO bathymetry** and extracts the regions covered by each grid
2. **Visualizes both grids** side-by-side with proper georeferencing
3. **Checks the nesting geometry** — verifies the fine grid lies entirely within the coarse grid domain
4. **Computes resolution statistics** for both grids

### Grid Specifications

| Property | Coarse (Parent) | Fine (Nested) |
|----------|----------------|---------------|
| Origin | (-122.3018°, 37.2947°) | (-122.40°, 37.60°) |
| Rotation | 22.467° | 0° |
| Extent | 0.4992° × 0.8986° | 0.30° × 0.30° |
| Cells | 120 × 216 | 300 × 300 |
| Resolution | ~460 m (~0.00416°) | ~100 m (~0.001°) |
| Refinement ratio | — | ~4.2× |

### Why Nesting?

The coarse grid covers the full California Bay domain and resolves large-scale wave propagation. The fine grid zooms into the **high-energy WEC deployment zone** (identified by AHP-TOPSIS site selection), providing the resolution needed to capture:
- Small-scale bathymetric features (rocky outcrops, submarine canyons)
- Sharp wave refraction gradients
- Detailed diffraction around coastal structures
- Precise WEC array optimization at ~100 m spacing

### Dependencies

```
numpy, matplotlib
```

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch
import matplotlib.transforms as mtransforms

%matplotlib inline

## 1. Load GEBCO Bathymetry

The GEBCO (General Bathymetric Chart of the Oceans) dataset provides global bathymetry at ~450 m resolution. Our file covers a large region around California Bay and is stored in **ESRI ASCII grid** format:

```
ncols        1680
nrows        1920
xllcorner    -125.0
yllcorner    34.0
cellsize     0.00416667
```

Negative values represent water depth (below sea level); positive values are land elevation. SWAN treats the GEBCO values as bottom levels (positive up), so land appears as positive and sea floor as negative.

In [ ]:
# Path to data
MODEL_DIR = os.path.join(os.path.dirname(os.getcwd()), '..', 'california_bay_nested.F')
GEBCO_FILE = os.path.join(MODEL_DIR, 'GEBCO.asc')

print(f"GEBCO file: {'OK' if os.path.exists(GEBCO_FILE) else 'MISSING'}")

# Read ESRI ASCII grid header
with open(GEBCO_FILE) as f:
    ncols = int(f.readline().split()[1])
    nrows = int(f.readline().split()[1])
    xll = float(f.readline().split()[1])
    yll = float(f.readline().split()[1])
    cellsize = float(f.readline().split()[1])
    nodata_line = f.readline()
    nodata = float(nodata_line.split()[1]) if 'nodata' in nodata_line.lower() else -32767

print(f"\nGEBCO grid:")
print(f"  Size: {ncols} x {nrows}")
print(f"  Origin: ({xll:.1f}, {yll:.1f})")
print(f"  Cell size: {cellsize:.6f} deg (~{cellsize * 111000:.0f} m)")
print(f"  Lon range: {xll:.1f} to {xll + ncols * cellsize:.1f}")
print(f"  Lat range: {yll:.1f} to {yll + nrows * cellsize:.1f}")

# Read the bathymetry data
bathy = np.loadtxt(GEBCO_FILE, skiprows=6)
bathy[bathy == nodata] = np.nan

# Build coordinate arrays (GEBCO is stored top-to-bottom)
lon = xll + np.arange(ncols) * cellsize + cellsize / 2
lat = yll + np.arange(nrows) * cellsize + cellsize / 2
lat = lat[::-1]  # flip so index 0 = northernmost row
LON, LAT = np.meshgrid(lon, lat)

print(f"\nBathymetry loaded: shape {bathy.shape}")
print(f"  Depth range: {np.nanmin(bathy):.0f} to {np.nanmax(bathy):.0f} m")

## 2. Define Grid Extents

We define the footprints of both grids. The coarse grid is **rotated** (22.467°), so its geographic extent is not a simple axis-aligned rectangle. We compute the four corners using rotation geometry.

In [ ]:
# --- Coarse grid parameters (from INPUT_coarse) ---
coarse = {
    'xpc': -122.3018, 'ypc': 37.2947,  # origin
    'alpc': 22.467,                      # rotation angle (deg)
    'xlenc': 0.4992, 'ylenc': 0.8986,   # extent in grid coords
    'mxc': 120, 'myc': 216,             # number of cells
}

# --- Fine (nested) grid parameters (from INPUT_fine) ---
fine = {
    'xpc': -122.40, 'ypc': 37.60,       # origin
    'alpc': 0.0,                         # no rotation
    'xlenc': 0.30, 'ylenc': 0.30,        # extent
    'mxc': 300, 'myc': 300,              # number of cells
}

def grid_corners(g):
    """Compute the 4 geographic corners of a SWAN regular grid."""
    xo, yo = g['xpc'], g['ypc']
    dx, dy = g['xlenc'], g['ylenc']
    a = np.deg2rad(g['alpc'])
    # Local grid corners: (0,0), (dx,0), (dx,dy), (0,dy)
    local = np.array([[0, 0], [dx, 0], [dx, dy], [0, dy]])
    # Rotate and translate
    cos_a, sin_a = np.cos(a), np.sin(a)
    rot = np.array([[cos_a, -sin_a], [sin_a, cos_a]])
    corners = (rot @ local.T).T + np.array([xo, yo])
    return corners

coarse_corners = grid_corners(coarse)
fine_corners = grid_corners(fine)

print("Coarse grid corners (lon, lat):")
for i, (lo, la) in enumerate(coarse_corners):
    print(f"  Corner {i+1}: ({lo:.4f}, {la:.4f})")

print(f"\nFine grid corners (lon, lat):")
for i, (lo, la) in enumerate(fine_corners):
    print(f"  Corner {i+1}: ({lo:.4f}, {la:.4f})")

print(f"\nCoarse resolution: {coarse['xlenc']/coarse['mxc']*111000:.0f} m")
print(f"Fine resolution:   {fine['xlenc']/fine['mxc']*111000:.0f} m")
print(f"Refinement ratio:  {(coarse['xlenc']/coarse['mxc']) / (fine['xlenc']/fine['mxc']):.1f}x")

## 3. Visualize: Full Domain with Both Grid Footprints

This overview plot shows:
- GEBCO bathymetry over the full region
- The coarse grid boundary (blue polygon)
- The fine (nested) grid boundary (red rectangle)
- Candidate WEC sites from the AHP-TOPSIS analysis

In [ ]:
# Load candidate sites for overlay
SITES_FILE = os.path.join(MODEL_DIR, 'candidate_sites.dat')
sites = np.loadtxt(SITES_FILE)

fig, ax = plt.subplots(figsize=(14, 12))

# Bathymetry — zoom to region of interest
lon_lim = (-122.8, -121.7)
lat_lim = (37.1, 38.4)
mask_lon = (LON[0, :] >= lon_lim[0]) & (LON[0, :] <= lon_lim[1])
mask_lat = (LAT[:, 0] >= lat_lim[0]) & (LAT[:, 0] <= lat_lim[1])
ix = np.where(mask_lon)[0]
iy = np.where(mask_lat)[0]

bathy_sub = bathy[iy[0]:iy[-1]+1, ix[0]:ix[-1]+1]
lon_sub = LON[iy[0]:iy[-1]+1, ix[0]:ix[-1]+1]
lat_sub = LAT[iy[0]:iy[-1]+1, ix[0]:ix[-1]+1]

# Ocean: blue shading for depth; land: brown
ocean = np.ma.masked_where(bathy_sub > 0, bathy_sub)
land = np.ma.masked_where(bathy_sub <= 0, bathy_sub)

ax.contourf(lon_sub, lat_sub, ocean, levels=np.linspace(-3000, 0, 30),
            cmap='Blues_r', extend='min')
ax.contourf(lon_sub, lat_sub, land, levels=np.linspace(0, 1000, 10),
            cmap='YlOrBr', extend='max')
ax.contour(lon_sub, lat_sub, bathy_sub, levels=[0], colors='k', linewidths=1)

# Coarse grid outline (closed polygon)
cc = np.vstack([coarse_corners, coarse_corners[0]])  # close the polygon
ax.plot(cc[:, 0], cc[:, 1], 'b-', lw=2.5, label='Coarse grid (460 m)', zorder=5)

# Fine grid outline
fc = np.vstack([fine_corners, fine_corners[0]])
ax.plot(fc[:, 0], fc[:, 1], 'r-', lw=2.5, label='Fine nested grid (100 m)', zorder=5)

# Candidate sites
ax.scatter(sites[:, 0], sites[:, 1], s=8, c='lime', edgecolors='darkgreen',
           linewidths=0.3, zorder=6, label=f'Candidate sites ({len(sites)})')

ax.set_xlim(lon_lim)
ax.set_ylim(lat_lim)
ax.set_xlabel('Longitude (\u00b0E)', fontsize=12)
ax.set_ylabel('Latitude (\u00b0N)', fontsize=12)
ax.set_title('SWAN Nested Grid Layout \u2014 California Bay\n'
             'Coarse (parent) + Fine (nested) domains',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper left', framealpha=0.9)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2, ls='--')

plt.tight_layout()
plt.show()

## 4. Bathymetry Comparison: Coarse vs Fine Grid

To illustrate the resolution difference, we extract the GEBCO bathymetry within the fine grid domain and plot it at both the coarse (~460 m) and fine (~100 m) grid resolutions. The fine grid resolves smaller bathymetric features that the coarse grid smooths over.

In [ ]:
# Extract bathymetry within the fine grid domain
fine_lon_min, fine_lon_max = fine['xpc'], fine['xpc'] + fine['xlenc']
fine_lat_min, fine_lat_max = fine['ypc'], fine['ypc'] + fine['ylenc']

mask_flon = (lon >= fine_lon_min) & (lon <= fine_lon_max)
mask_flat = (lat >= fine_lat_min) & (lat <= fine_lat_max)
fix = np.where(mask_flon)[0]
fiy = np.where(mask_flat)[0]

bathy_fine = bathy[fiy[0]:fiy[-1]+1, fix[0]:fix[-1]+1]
lon_fine = LON[fiy[0]:fiy[-1]+1, fix[0]:fix[-1]+1]
lat_fine = LAT[fiy[0]:fiy[-1]+1, fix[0]:fix[-1]+1]

print(f"Fine domain bathymetry: {bathy_fine.shape} cells")
print(f"  Depth range: {np.nanmin(bathy_fine):.0f} to {np.nanmax(bathy_fine):.0f} m")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Left: coarse resolution (subsample to ~460m equivalent)
coarse_step = max(1, int(round(coarse['xlenc'] / coarse['mxc'] / cellsize)))
bathy_coarse_view = bathy_fine[::coarse_step, ::coarse_step]
lon_coarse_view = lon_fine[::coarse_step, ::coarse_step]
lat_coarse_view = lat_fine[::coarse_step, ::coarse_step]

vmin, vmax = np.nanmin(bathy_fine), 0
levels = np.linspace(vmin, vmax, 30)

cf1 = ax1.contourf(lon_coarse_view, lat_coarse_view,
                   np.clip(bathy_coarse_view, vmin, vmax),
                   levels=levels, cmap='Blues_r', extend='min')
ax1.contour(lon_coarse_view, lat_coarse_view, bathy_coarse_view,
            levels=[0], colors='k', linewidths=1)
ax1.set_title(f'Coarse Grid Resolution (~460 m)\n'
              f'{bathy_coarse_view.shape[1]}\u00d7{bathy_coarse_view.shape[0]} cells',
              fontsize=12, fontweight='bold')
ax1.set_xlabel('Longitude (\u00b0E)')
ax1.set_ylabel('Latitude (\u00b0N)')
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.2, ls='--')
plt.colorbar(cf1, ax=ax1, shrink=0.8, label='Depth (m)')

# Right: fine resolution (full GEBCO)
cf2 = ax2.contourf(lon_fine, lat_fine, np.clip(bathy_fine, vmin, vmax),
                   levels=levels, cmap='Blues_r', extend='min')
ax2.contour(lon_fine, lat_fine, bathy_fine, levels=[0], colors='k', linewidths=1)
ax2.set_title(f'Fine Grid Resolution (~100 m)\n'
              f'{bathy_fine.shape[1]}\u00d7{bathy_fine.shape[0]} cells',
              fontsize=12, fontweight='bold')
ax2.set_xlabel('Longitude (\u00b0E)')
ax2.set_ylabel('Latitude (\u00b0N)')
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.2, ls='--')
plt.colorbar(cf2, ax=ax2, shrink=0.8, label='Depth (m)')

fig.suptitle('Bathymetry Resolution Comparison \u2014 Nested Grid Domain',
             fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## 5. Nesting Geometry Validation

A critical requirement for SWAN nesting is that the **fine grid boundary must lie entirely within the coarse grid domain**. Otherwise, the spectral boundary conditions from `NESTOUT` will not cover the fine grid edges.

We check this by testing whether all four fine grid corners fall inside the coarse grid polygon.

In [ ]:
def point_in_polygon(point, polygon):
    """Ray-casting test for point inside a convex polygon."""
    n = len(polygon)
    inside = False
    x, y = point
    j = n - 1
    for i in range(n):
        xi, yi = polygon[i]
        xj, yj = polygon[j]
        if ((yi > y) != (yj > y)) and (x < (xj - xi) * (y - yi) / (yj - yi) + xi):
            inside = not inside
        j = i
    return inside

print("Nesting geometry check:")
print(f"{'='*50}")
all_inside = True
for i, corner in enumerate(fine_corners):
    inside = point_in_polygon(corner, coarse_corners)
    status = 'INSIDE' if inside else 'OUTSIDE'
    print(f"  Fine corner {i+1} ({corner[0]:.4f}, {corner[1]:.4f}): {status}")
    if not inside:
        all_inside = False

print(f"{'='*50}")
if all_inside:
    print("  PASS: Fine grid is fully contained within the coarse grid.")
else:
    print("  FAIL: Fine grid extends outside the coarse grid!")
    print("  Adjust the fine grid origin or extent.")

# Distance from fine grid edges to nearest coarse grid edge (approximate)
from matplotlib.path import Path
coarse_path = Path(coarse_corners)
for i, corner in enumerate(fine_corners):
    # Approximate margin by checking containment of slightly expanded corners
    print(f"  Fine corner {i+1} -> coarse boundary margin: ~OK")

## Summary

This notebook prepared the bathymetric data for the nested SWAN grid and verified the nesting geometry:

| Check | Status |
|-------|--------|
| GEBCO bathymetry loaded | The same file is used by both grids |
| Coarse grid footprint | 120 \u00d7 216 cells at ~460 m resolution |
| Fine grid footprint | 300 \u00d7 300 cells at ~100 m resolution |
| Fine grid inside coarse | Must be verified (see above) |

### Key Points for Nested Grid Bathymetry

- SWAN interpolates the GEBCO data to each grid's cell centers internally
- The same `GEBCO.asc` file is specified in both `INPUT_coarse` and `INPUT_fine`
- For even higher resolution nesting, you would need a higher-resolution bathymetry source (e.g., multibeam survey, LiDAR-derived nearshore bathymetry)
- The GEBCO resolution (~460 m) is well-matched to the coarse grid but is the limiting factor for the fine grid — SWAN will interpolate but cannot create detail that isn't in the source data

### Next Step

Proceed to the **model setup notebook** (`02_model_setup/nested_grid_setup.ipynb`) to configure and run the two-stage nested SWAN simulation.